# Beekeeper — Notebook 02: Preprocesamiento

**Proyecto Final IA** — Curso de Especialización en Inteligencia Artificial y Big Data — IES Azarquiel

---

## Qué hace este notebook

Ejecuta las decisiones tomadas en el notebook 01 (EDA):

1. Carga el CSV de etiquetas y enlaza cada uno de los ~7.100 segmentos de 60s con su audio original.
2. Extrae el **Mel-espectrograma** de cada segmento (`n_mels=128`, `n_fft=2048`, `hop_length=512`) y lo pasa a **escala dB** con `librosa.power_to_db`.
3. **Pad / truncate** todos los espectrogramas a la misma longitud temporal `T_FIXED` para poder apilarlos en un tensor.
4. **Z-score normaliza** cada espectrograma (media 0, std 1) → ayuda muchísimo a que el LSTM entrene rápido.
5. Guarda todo en un único `.npz` en Drive: `X` (espectrogramas), `y` (clases 0-3), `groups` (audio original) y `segmentos` (nombres).
6. Genera los splits **GroupKFold** agrupando por audio original — los segmentos de una misma grabación nunca se reparten entre train y val.

El notebook tarda un rato (~10-20 min según Drive); el resultado son ficheros que el notebook 03 (LSTM) carga en segundos.

## Configuración

In [ ]:
# Notebook adaptado para ejecucion local (sin Google Colab).
# Las rutas se resuelven automaticamente en la celda de setup a partir de ENTREGA/data/
# (donde debe estar el dataset de Kaggle; ver descargar_datos.py).


In [ ]:
!pip install -q librosa

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

from pathlib import Path
_cand = [Path.cwd()/'data', Path.cwd().parent/'data', Path.cwd().parent/'ENTREGA'/'data',
         Path.cwd()/'ENTREGA'/'data', Path(r'D:/abejas/ENTREGA/data')]
DATA_DIR = next((p for p in _cand if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("No encuentro la carpeta 'data' con el dataset. "
                            "Descargalo en ENTREGA/data/ (ver descargar_datos.py).")
ENTREGA_DIR = DATA_DIR.parent
_csv = next(iter(DATA_DIR.rglob('all_data_updated.csv')), None)
if _csv is None:
    raise FileNotFoundError(f"No encuentro 'all_data_updated.csv' bajo {DATA_DIR}.")
RUTA_BASE   = str(_csv.parent) + os.sep
RUTA_AUDIOS = next((str(p) + os.sep for p in [_csv.parent/'sound_files'/'sound_files', _csv.parent/'sound_files'] if p.is_dir()), RUTA_BASE)
RUTA_OUT = str(DATA_DIR) + os.sep
os.makedirs(RUTA_OUT, exist_ok=True)

# Parámetros del Mel-espectrograma (idénticos a los del EDA)
SR = 22050
N_MELS = 128
N_FFT = 2048
HOP = 512
DURACION_SEG = 60.0  # cada segmento dura 60 s

# Longitud temporal fija a la que se trunca/padea cada espectrograma
T_FIXED = int(np.floor(DURACION_SEG * SR / HOP)) + 1   # ≈ 2585 frames

print(f'T_FIXED = {T_FIXED} frames temporales por segmento')
print(f'Cada espectrograma → shape ({N_MELS}, {T_FIXED})')
print('RUTA_BASE  :', RUTA_BASE)
print('RUTA_AUDIOS:', RUTA_AUDIOS)


## 1. Construcción del DataFrame de trabajo

Repetimos el join `segmento → audio original → CSV` que ya validamos en el notebook 01.
El resultado es un DataFrame `df` de ~7.100 filas con la etiqueta `queen status` y el grupo (`file name`).

In [ ]:
main_dataset = pd.read_csv(RUTA_BASE + 'all_data_updated.csv')

archivos_en_disco = sorted(os.listdir(RUTA_AUDIOS))


def segmento_a_original(nombre_segmento: str) -> str:
    return re.sub(r'__segment\d+\.wav$', '.raw', nombre_segmento)


df_segmentos = pd.DataFrame({'segmento': archivos_en_disco})
df_segmentos['file name'] = df_segmentos['segmento'].apply(segmento_a_original)

df = df_segmentos.merge(main_dataset, on='file name', how='left')
df = df.dropna(subset=['queen status']).reset_index(drop=True)
df['queen status'] = df['queen status'].astype(int)

print(f'Segmentos a procesar: {len(df)}')
print(f'Distribución de clases:')
print(df['queen status'].value_counts().sort_index())

## 2. Extracción de Mel-espectrogramas

Función `audio_a_mel`: carga el `.wav`, calcula el Mel-espectrograma, pasa a dB y normaliza por z-score.
Después se aplica a cada uno de los ~7.100 segmentos.

In [ ]:
def audio_a_mel(ruta: str) -> np.ndarray:
    """Carga audio, devuelve Mel-espectrograma en dB normalizado a (N_MELS, T_FIXED)."""
    y, _ = librosa.load(ruta, sr=SR, mono=True)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # (N_MELS, T)

    # Pad o truncate temporal a T_FIXED
    T = mel_db.shape[1]
    if T < T_FIXED:
        mel_db = np.pad(mel_db, ((0, 0), (0, T_FIXED - T)),
                        mode='constant', constant_values=mel_db.min())
    elif T > T_FIXED:
        mel_db = mel_db[:, :T_FIXED]

    # Z-score por espectrograma (centra y escala — el LSTM lo agradece)
    mu, sigma = mel_db.mean(), mel_db.std() + 1e-6
    mel_db = (mel_db - mu) / sigma

    return mel_db.astype(np.float32)

In [ ]:
# Reservamos el tensor completo de antemano para no fragmentar memoria
N = len(df)
X = np.zeros((N, N_MELS, T_FIXED), dtype=np.float32)
y = df['queen status'].values.astype(np.int64)
groups = df['file name'].values  # audio original — lo usamos en GroupKFold
segmentos = df['segmento'].values

errores = []
for i, nombre in enumerate(tqdm(segmentos, desc='Extrayendo Mel')):
    try:
        X[i] = audio_a_mel(os.path.join(RUTA_AUDIOS, nombre))
    except Exception as e:
        errores.append((nombre, str(e)))

print(f'\nProcesados: {N - len(errores)} / {N}  | errores: {len(errores)}')
if errores[:3]:
    print('Primeros errores:', errores[:3])

## 3. Splits train / val con GroupKFold

`GroupKFold(n_splits=5)` agrupa por `file name` (audio original). Esto garantiza que **ninguna grabación quede repartida** entre train y val: los 5-6 segmentos de una grabación van todos al mismo lado. Sin esto, el modelo memoriza la firma acústica de la grabación en lugar del patrón de la clase.

Guardamos los índices del fold 0 como split por defecto, pero también guardamos los 5 folds por si queremos hacer cross-validation más adelante.

In [ ]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, groups=groups))

train_idx, val_idx = folds[0]

print(f'Train: {len(train_idx)} segmentos | Val: {len(val_idx)} segmentos')
print(f'Audios originales en train: {pd.Series(groups[train_idx]).nunique()}')
print(f'Audios originales en val:   {pd.Series(groups[val_idx]).nunique()}')

# Sanity: ningún audio original aparece en los dos
inter = set(groups[train_idx]) & set(groups[val_idx])
assert len(inter) == 0, f'¡Leakage! {len(inter)} audios en ambos splits'
print('OK — sin leakage de audios originales entre splits')

# Distribución de clases en cada split
print('\nClases en train:', dict(zip(*np.unique(y[train_idx], return_counts=True))))
print('Clases en val:  ', dict(zip(*np.unique(y[val_idx],   return_counts=True))))

## 4. Guardado en disco

In [ ]:
ruta_npz = os.path.join(RUTA_OUT, 'mel_dataset.npz')

np.savez_compressed(
    ruta_npz,
    X=X,
    y=y,
    groups=groups,
    segmentos=segmentos,
    train_idx=train_idx,
    val_idx=val_idx,
    sr=SR,
    n_mels=N_MELS,
    hop=HOP,
    t_fixed=T_FIXED,
)

print(f'Guardado: {ruta_npz}')
print(f'Tamaño: {os.path.getsize(ruta_npz) / 1e6:.1f} MB')

# Guardamos también todos los folds por si luego hacemos CV
np.savez(
    os.path.join(RUTA_OUT, 'folds.npz'),
    **{f'fold{i}_train': tr for i, (tr, _) in enumerate(folds)},
    **{f'fold{i}_val':   va for i, (_, va) in enumerate(folds)},
)
print(f'Guardado: {os.path.join(RUTA_OUT, "folds.npz")}')

---

## 5. Verificación rápida

Cargamos el `.npz` desde cero y comprobamos un par de cosas: shape, rango de valores tras la normalización, y que las etiquetas siguen cuadrando.

In [ ]:
data = np.load(ruta_npz, allow_pickle=True)
print('Claves guardadas:', list(data.keys()))
print(f'X shape: {data["X"].shape}  dtype: {data["X"].dtype}')
print(f'y shape: {data["y"].shape}  clases únicas: {np.unique(data["y"])}')
print(f'X media≈{data["X"].mean():.3f}  std≈{data["X"].std():.3f}  '
      f'(esperado: media≈0, std≈1)')
print(f'rango: [{data["X"].min():.2f}, {data["X"].max():.2f}]')

El `.npz` queda listo para que el notebook **`03_lstm.ipynb`** lo cargue y entrene el modelo.